# Proof of Draw

A creator announces ticket **#4** as the winner of a fictional $1 million giveaway. Eight subscribers each hold one numbered ticket. How can a viewer check the announcement? A public receipt supplies the ordered ticket list, a fixed selection rule, and an archived number from the University of Colorado Boulder’s randomness beacon. The claim to test is **which ticket the receipt selects**.

**Run the notebook.** Keep `curbyq-round-28297.json` beside this notebook and run the cells in order. In Colab, upload the JSON through the Files panel. The receipt uses the Python standard library. The experiment uses Qiskit and NumPy; setup instructions are in the README.

**Before a live giveaway.** Publish who is eligible, the final ordered ticket list, the selection rule and the exact future beacon round before the outcome is available. Publish a fallback rule for an unavailable round at the same time. This replay uses fictional entrants and an already published record.

[CU Boulder’s archived round](https://random.colorado.edu/api/curbyq/round/28297) · [NIST on advance commitments for public draws](https://csrc.nist.gov/projects/interoperable-randomness-beacons/apps)

## 1. Read the receipt

The archive contains several records about round 28297. The `randomness` record contains the final 512-bit pulse encoded as base64. The ticket list is ordered #1 through #8. Read the **first three bits**, treating the leftmost bit as the highest-value bit. Use the resulting number, 0 through 7, as the position in the list, starting at zero.

In [1]:
import base64
import hashlib
import json
from pathlib import Path

archive = Path("curbyq-round-28297.json")
snapshot = archive.read_bytes()
records = json.loads(snapshot)
round_number = 28297
round_url = f"https://random.colorado.edu/api/curbyq/round/{round_number}"
tickets = tuple(range(1, 9))

payloads = [record["data"]["content"]["payload"] for record in records]
finished = [p for p in payloads if p["round"] == round_number and p["stage"] == "randomness"]
if len(finished) != 1:
    raise ValueError("Expected exactly one completed randomness record for the selected round.")
record = finished[0]
encoded = record["randomness"]["/"]["bytes"]
pulse = base64.b64decode(encoded + "=" * (-len(encoded) % 4), validate=True)

def select_ticket(pulse_bytes, ticket_list):
    if len(pulse_bytes) != 64:
        raise ValueError("Expected a 512-bit beacon pulse.")
    if len(ticket_list) != 8 or len(set(ticket_list)) != 8:
        raise ValueError("This rule requires eight distinct tickets in a fixed order.")
    return ticket_list[pulse_bytes[0] >> 5]

winner = select_ticket(pulse, tickets)
print(f"Source: {round_url}")
print(f"Published (UTC): {record['timestamp']}")
print("Ordered tickets:", ", ".join(f"#{ticket}" for ticket in tickets))
print(f"Pulse length: {len(pulse) * 8} bits")
print(f"First byte: 0x{pulse[0]:02X} = {pulse[0]:08b}")
print(f"First three bits: {pulse[0] >> 5:03b} = position {pulse[0] >> 5} (zero-based)")
print(f"Computed winner: ticket #{winner}")
print(f"Archive file SHA-256: {hashlib.sha256(snapshot).hexdigest()}")

Source: https://random.colorado.edu/api/curbyq/round/28297
Published (UTC): 2025-08-22T19:42:38.156Z
Ordered tickets: #1, #2, #3, #4, #5, #6, #7, #8
Pulse length: 512 bits
First byte: 0x61 = 01100001
First three bits: 011 = position 3 (zero-based)
Computed winner: ticket #4
Archive file SHA-256: b70ec1f7c6db0bcf09308340d14052f40ad1ce119803281d6c8f8563dd088d58


## 2. Try changing the announcement

A screenshot naming a winner gives the audience one claim and no independent input with which to check it. Here the reference result is the ticket calculated from the public record. Change `announced_ticket` below and rerun the cell. The archive, the ticket list and the rule stay fixed across claims.

In [2]:
def verify_claim(announced, pulse_bytes, ticket_list):
    return announced == select_ticket(pulse_bytes, ticket_list)

announced_ticket = 4
for label, claim in (("Your claim", announced_ticket), ("Altered announcement", 7)):
    matches = verify_claim(claim, pulse, tickets)
    print(f"{label}: ticket #{claim} | {'MATCH' if matches else 'MISMATCH'} | receipt: #{winner}")

Your claim: ticket #4 | MATCH | receipt: #4
Altered announcement: ticket #7 | MISMATCH | receipt: #4


### Check every possible ticket

All 256 first bytes are possible inputs to the three-bit rule. Each group of 32 consecutive bytes has the same prefix. We test every byte against its expected ticket and exercise the checker with all eight claims for the actual receipt. Equal chances in a future draw require equally likely prefixes and a properly fixed list. These tests measure the implemented mapping.

In [3]:
from collections import Counter

expected_prefixes = ("000", "001", "010", "011", "100", "101", "110", "111")
prefix_counts = Counter()
for first_byte in range(256):
    fixture = bytes([first_byte]) + bytes(63)
    ticket = select_ticket(fixture, tickets)
    prefix = f"{first_byte:08b}"[:3]
    assert prefix == expected_prefixes[ticket - 1]
    prefix_counts[ticket] += 1

for ticket, prefix in zip(tickets, expected_prefixes):
    assert prefix_counts[ticket] == 32
    print(f"{prefix} → ticket #{ticket} ({prefix_counts[ticket]}/256 first bytes)")

matches = [ticket for ticket in tickets if verify_claim(ticket, pulse, tickets)]
assert matches == [4] and not verify_claim(7, pulse, tickets)
print(f"Archived round: 1 matching claim (#{matches[0]}) and 7 rejected claims.")

000 → ticket #1 (32/256 first bytes)
001 → ticket #2 (32/256 first bytes)
010 → ticket #3 (32/256 first bytes)
011 → ticket #4 (32/256 first bytes)
100 → ticket #5 (32/256 first bytes)
101 → ticket #6 (32/256 first bytes)
110 → ticket #7 (32/256 first bytes)
111 → ticket #8 (32/256 first bytes)
Archived round: 1 matching claim (#4) and 7 rejected claims.


## 3. Why a quantum randomness beacon?

A classical public randomness source can also give viewers the inputs to replay a draw. The quantum part addresses an earlier question. What evidence supports the claim that the source’s bits were unpredictable **before** publication? CU Boulder and NIST use physical Bell-test data, statistical analysis and randomness extraction. Our calculator uses their published output.

The CHSH game below illustrates Bell correlations on a **classical computer running Qiskit's ideal simulator**. Two separated players, Alice and Bob, receive one private bit each. They agree on a strategy before receiving their bits and cannot communicate afterward. They win if their answers match, except when both questions are 1, when their answers must differ. The metric is the fraction of wins, with equal weight for all four question pairs.

[IBM Quantum’s CHSH game lesson](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/entanglement-in-action/chsh-game) · [CU Boulder/NIST beacon experiment](https://doi.org/10.1038/s41586-025-09054-3)

In [4]:
import math
from itertools import product

import numpy as np
import qiskit
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

print("Qiskit", qiskit.__version__, "| NumPy", np.__version__)

Qiskit 1.4.5 | NumPy 2.0.2


### Classical baseline

A classical plan specifies Alice’s answer to 0 and 1, and Bob’s answer to 0 and 1. The four yes-or-no choices yield 16 plans. Shared randomness amounts to choosing a plan by chance, so its average win rate cannot exceed the best fixed plan. This baseline assumes the four joint questions are equally likely.

In [5]:
questions = list(product((0, 1), repeat=2))

def wins(x, y, alice_answer, bob_answer):
    return (alice_answer != bob_answer) == (x == 1 and y == 1)

def classical_rate(alice, bob):
    return sum(wins(x, y, alice[x], bob[y]) for x, y in questions) / len(questions)

alice_plan = (0, 0)  # Answer for input 0, then answer for input 1
bob_plan = (0, 0)
for x, y in questions:
    print(f"Questions {x},{y}: {'win' if wins(x,y,alice_plan[x],bob_plan[y]) else 'lose'}")
rates = [classical_rate(bits[:2], bits[2:]) for bits in product((0, 1), repeat=4)]
best_classical = max(rates)
print(f"Your strategy: {classical_rate(alice_plan, bob_plan):.0%}")
print(f"Highest possible among all {len(rates)} classical plans: {best_classical:.0%}")
assert len(rates) == 16 and best_classical == 0.75

Questions 0,0: win
Questions 0,1: win
Questions 1,0: win
Questions 1,1: lose
Your strategy: 75%
Highest possible among all 16 classical plans: 75%


### Entangled strategy

Qiskit prepares a two-qubit Bell pair with an H gate followed by a controlled X gate. Each player’s question selects a measurement direction for their own qubit. We read the pairs of answers in 2,000 simulated trials per question pair. The exact probabilities come from the circuit state, and the measured counts come from sampling it. The four reproducible seeds are 101 through 104.

In [6]:
def shared_pair():
    circuit = QuantumCircuit(2)
    circuit.h(0)
    circuit.cx(0, 1)
    return circuit

def game_circuit(x, y):
    circuit = shared_pair()
    circuit.barrier()
    circuit.ry(-x * math.pi / 2, 0)  # Alice's direction depends only on x
    bob_direction = math.pi / 4 if y == 0 else -math.pi / 4
    circuit.ry(-bob_direction, 1)
    return circuit

trials_per_pair = 2000
states = {}
ideal_rates = []
win_count = 0
for index, (x, y) in enumerate(questions):
    state = Statevector.from_instruction(game_circuit(x, y))
    states[x, y] = state
    # Qiskit labels put qubit 0 (Alice) on the right.
    winning_labels = [label for label in ("00", "01", "10", "11")
                      if wins(x, y, int(label[1]), int(label[0]))]
    probabilities = state.probabilities_dict()
    ideal_rates.append(sum(probabilities.get(label, 0) for label in winning_labels))
    state.seed(101 + index)
    counts = state.sample_counts(trials_per_pair)
    wins_for_pair = sum(counts.get(label, 0) for label in winning_labels)
    win_count += wins_for_pair
    print(f"Questions {x},{y}: {wins_for_pair}/{trials_per_pair} wins")
ideal_rate = sum(ideal_rates) / len(ideal_rates)
total = len(questions) * trials_per_pair
print(f"Sampled quantum strategy: {win_count}/{total} = {win_count / total:.2%}")
print(f"Ideal circuit probability: {ideal_rate:.2%}")
print(f"Best classical strategy: {best_classical:.2%}")
assert np.allclose(ideal_rates, math.cos(math.pi / 8) ** 2)

Questions 0,0: 1709/2000 wins
Questions 0,1: 1716/2000 wins
Questions 1,0: 1694/2000 wins
Questions 1,1: 1726/2000 wins
Sampled quantum strategy: 6845/8000 = 85.56%
Ideal circuit probability: 85.36%
Best classical strategy: 75.00%


### Sampling uncertainty

Each question pair has the same ideal win probability in this circuit. The Wilson interval below estimates the uncertainty due to sampling a finite number of trials. Its interpretation assumes independent trials. The fixed seeds make this particular simulated sample reproducible.

In [7]:
from statistics import NormalDist

z = NormalDist().inv_cdf(0.975)
observed = win_count / total
denominator = 1 + z**2 / total
center = (observed + z**2 / (2 * total)) / denominator
half_width = z * math.sqrt(observed * (1 - observed) / total + z**2 / (4 * total**2)) / denominator
print(f"Observed: {observed:.2%}; approximate 95% Wilson interval: "
      f"{center - half_width:.2%} to {center + half_width:.2%}")
print(f"Exact ideal probability: {ideal_rate:.2%}")

Observed: 85.56%; approximate 95% Wilson interval: 84.78% to 86.32%
Exact ideal probability: 85.36%


### What happens when an answer is recorded incorrectly?

In a simple sensitivity test, each of the two answers flips independently with probability `p`. Exactly one flip reverses the game outcome. Fresh trials in each row introduce sampling variation. The last column reports the model’s exact expectation, which is the appropriate quantity for finding its crossover with the error-free classical limit.

In [8]:
error_rates = [0.00, 0.04, 0.07, 0.08, 0.10, 0.15]
rng = np.random.default_rng(204)
print("Error per answer | Simulated wins | Expected wins")
for p in error_rates:
    assert 0 <= p <= 0.5, "Choose error probabilities from 0 to 0.5."
    total_wins = 0
    for x, y in questions:
        outcomes = rng.choice(4, size=trials_per_pair, p=states[x, y].probabilities())
        alice = (outcomes % 2) ^ (rng.random(trials_per_pair) < p)
        bob = (outcomes // 2) ^ (rng.random(trials_per_pair) < p)
        total_wins += np.count_nonzero(wins(x, y, alice, bob))
    one_flip = 2 * p * (1 - p)
    expected = ideal_rate * (1 - one_flip) + (1 - ideal_rate) * one_flip
    print(f"{p:>15.1%} | {total_wins / total:>14.2%} | {expected:>13.2%}")
threshold = (1 - math.sqrt((best_classical - 0.5) / (ideal_rate - 0.5))) / 2
print(f"Expected win rate meets the classical limit at {threshold:.2%} error per answer.")
assert math.isclose(threshold, (1 - 2 ** (-0.25)) / 2, abs_tol=1e-12)

Error per answer | Simulated wins | Expected wins
           0.0% |         85.92% |        85.36%
           4.0% |         80.45% |        79.92%
           7.0% |         74.99% |        76.15%
           8.0% |         74.79% |        74.95%
          10.0% |         73.10% |        72.63%
          15.0% |         67.50% |        67.32%
Expected win rate meets the classical limit at 7.96% error per answer.


## What the evidence supports

- **Receipt.** Anyone with the same ordered list, rule and archived pulse can recompute ticket #4 and detect a claim of #7. A viewer can compare the bundled pulse with the public archive. The SHA-256 value identifies this copy of the data. A file hash alone cannot authenticate its publisher.
- **Eligibility and timing.** A live organizer must establish that the list is complete, every eligible entrant has the promised number of tickets, and the commitment predates the pulse. This fictional retrospective draw supplies no evidence about those steps or any payout. The three-bit rule is specific to exactly eight tickets.
- **Quantum experiment.** The sampled score and noise threshold illustrate a theoretical benchmark. This simulation used no physical quantum hardware. It neither produced the archived pulse nor independently verified CU Boulder’s extraction procedure, physical measurements or certificates. Its interval covers sampling uncertainty inside the ideal model. The noise test assumes independent symmetric recording errors.

## Sources and contributions

The physical beacon and its published data are the work of CU Boulder, NIST and their collaborators. The classical bound and optimal CHSH circuit are established results adapted from IBM Quantum Learning. This project applies them to an eight-ticket receipt checker and adds exhaustive mapping checks, a false-announcement test and an independent-readout-error sensitivity test. OpenAI Codex assisted with implementation, documentation and execution checks.

[CU Boulder round 28297](https://random.colorado.edu/api/curbyq/round/28297) · [Beacon research paper](https://doi.org/10.1038/s41586-025-09054-3) · [IBM circuit lesson](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/entanglement-in-action/qiskit-implementation) · [NIST public-draw examples](https://csrc.nist.gov/projects/interoperable-randomness-beacons/apps)